# The standard library

Python comes with about 200 modules ("batteries included"). You've already used quite a few:

| Module | Where |
|---|---|
| `sys` | week 1 |
| `math`, `copy`, `collections` (`Counter`, `defaultdict`) | week 3 |
| `pathlib`, `csv`, `json`, `unicodedata` | week 4 |
| `time` (`perf_counter`) | weeks 5 and 6 |
| `functools` (`wraps`, `lru_cache`) | Nov 11, Nov 18 |
| `dataclasses` | week 6 |
| `random` | Nov 18 |

Today: more of it, picked for what you'll need in the project. Rule of
thumb: before you write a general-purpose helper, check whether it exists.

All of it is documented at https://docs.python.org/3/library/.

## `itertools`: loops you don't have to write

Ex 2 Part 2 tried every noun and every verb from 0 to 99, with two nested loops. `itertools.product` makes those pairs for you:

In [ ]:
import itertools

print(list(itertools.product(range(3), repeat=2)))

The same pairs, in the same order, as `for a in range(3):` / `for b in range(3):`. With two
different iterables:

In [ ]:
print(list(itertools.product("AB", [1, 2, 3])))

**Predict:** how many pairs did Ex 2 try?

In [ ]:
print(len(list(itertools.product(range(100), repeat=2))))

Does order matter? `permutations` says yes, `combinations` says no. **Predict:** which line is
longer?

In [ ]:
print(list(itertools.permutations("ABC", 2)))
print(list(itertools.combinations("ABC", 2)))

`combinations(..., 2)` is "every pair once": every two weather stations to compare, every two
teams that play each other. `math` counts them without making them:

In [ ]:
import math

print(math.perm(3, 2), math.comb(3, 2))
print(math.comb(18, 2))  # 18 Bundesliga clubs: how many pairings?

`pairwise` gives neighbours, each item with the one after it:

In [ ]:
temperatures = [4.1, 5.3, 2.0, 2.0, 6.5]
print(list(itertools.pairwise(temperatures)))

**Predict:** the change from one day to the next:

In [ ]:
changes = [round(b - a, 1) for a, b in itertools.pairwise(temperatures)]
print(changes)

`accumulate` gives running totals, `chain` one iterable after the other. **Predict:**

In [ ]:
rain = [0.0, 2.5, 0.0, 1.2, 4.0]
print(list(itertools.accumulate(rain)))
print(list(itertools.chain([1, 2], (3, 4), "ab")))

### Lazy

Why `list(...)` around everything? Without it:

In [ ]:
pairs = itertools.product(range(100), repeat=2)
print(pairs)

In [ ]:
import time

t0 = time.perf_counter()
everything = itertools.product(range(100_000), repeat=2)
print(f"{time.perf_counter() - t0:.6f} s")

Ten billion pairs in a few milliseconds: none of them has been made yet. `product` hands out one
pair each time a loop asks for the next. **Predict:**

In [ ]:
for a, b in everything:
    if a + b == 3:
        print(a, b)
        break

Run it: what happens the second time?

In [ ]:
pairs = itertools.product(range(2), repeat=2)
print(list(pairs))
print(list(pairs))

Once a pair has been handed out, it's gone. If you need the values twice, make a list once and
use that.

## `operator`: operators as functions

IntCode: `if opcode == 1: ... elif opcode == 2: ...`. Every operator also exists as a function:

In [ ]:
import operator

print(operator.add(2, 5), operator.mul(2, 5), operator.lt(2, 5))

Functions are values (week 5), so they can go into a dict. **Predict:**

In [ ]:
OPERATIONS = {1: operator.add, 2: operator.mul, 7: operator.lt, 8: operator.eq}

opcode, a, b = 2, 6, 7
print(OPERATIONS[opcode](a, b))
print(OPERATIONS[7](a, b))

You've sorted with `key=lambda ...`. For "the item at this index" (or key), `itemgetter` makes that
function for you:

In [ ]:
stations = [("Mannheim", 11.2), ("Feldberg", 3.9), ("Freiburg", 11.9)]

print(sorted(stations, key=lambda station: station[1]))
print(sorted(stations, key=operator.itemgetter(1)))

**Predict:**

In [ ]:
get_first = operator.itemgetter(0)
print(get_first("Mannheim"))
print(max(stations, key=operator.itemgetter(1)))

`operator.attrgetter("height")` does the same for attributes, e.g. a list of dataclass objects.
Use whichever reads better to you; the `lambda` is fine too.

## `collections.deque` and `heapq`: queues

A *breadth-first search* finds the fewest steps through a maze by spreading out like a wave:
first every position one step away, then every position two steps away, and so on. It keeps a
queue of positions still to look at, and takes them from the front: `queue.pop(0)`. After that,
every remaining item in the list moves one place to the left. A `deque` ("deck") is built for
taking from both ends.

In [ ]:
from collections import deque


def time_list(n):
    items = list(range(n))
    t0 = time.perf_counter()
    while items:
        items.pop(0)
    return time.perf_counter() - t0


def time_deque(n):
    items = deque(range(n))
    t0 = time.perf_counter()
    while items:
        items.popleft()
    return time.perf_counter() - t0


print(f"list:  {time_list(100_000):.3f} s")
print(f"deque: {time_deque(100_000):.3f} s")

**Predict:** twice as many items. How much longer does each one take?

In [ ]:
print(f"list:  {time_list(200_000):.3f} s")
print(f"deque: {time_deque(200_000):.3f} s")

The method names say which end:

In [ ]:
queue = deque([1, 2, 3])
queue.append(4)
queue.appendleft(0)
print(queue)
print(queue.popleft(), queue.pop())
print(queue)

In a breadth-first search: `queue = deque([start])` and `queue.popleft()`.

### `heapq`: smallest first

Breadth-first search takes the *oldest* position first. Ex 5 Part 2 (lowest risk, any direction)
needs the position with the *lowest risk so far* first: a priority queue. `heapq` keeps a plain
list arranged so that the smallest item is always at index 0.

In [ ]:
import heapq

frontier = []
heapq.heappush(frontier, 5)
heapq.heappush(frontier, 1)
heapq.heappush(frontier, 3)
print(frontier[0])

`heappop` takes the smallest item out, and the heap rearranges itself. **Predict:**

In [ ]:
print(heapq.heappop(frontier), heapq.heappop(frontier))
print(frontier)

It's not a sorted list, just good enough that the smallest is in front:

In [ ]:
heap = []
for number in [5, 1, 3, 0, 4, 2]:
    heapq.heappush(heap, number)
print(heap)

With tuples `(risk, position)`, tuples are compared item by item, like strings letter by letter.
**Predict** the order:

In [ ]:
frontier = []
heapq.heappush(frontier, (7, (0, 2)))
heapq.heappush(frontier, (3, (1, 1)))
heapq.heappush(frontier, (3, (0, 4)))
while frontier:
    print(heapq.heappop(frontier))

Finding the smallest with `min(...)` over all candidates means looking at every one of them, every
step. With a heap, push and pop stay cheap however long the list gets. For "the 2 smallest" once,
there's also:

In [ ]:
print(heapq.nsmallest(2, stations, key=operator.itemgetter(1)))

## `re`: patterns in text

`split` gets you far with puzzle inputs and data files. When lines vary, a *regular expression*
describes what you're looking for:

| Pattern | Matches |
|---|---|
| `\d` | one digit |
| `\w` | one letter, digit or `_` |
| `\s` | one space, tab or newline |
| `.` | any one character |
| `+`, `*`, `?` | the thing before: 1 or more, 0 or more, 0 or 1 times |
| `{5}` | the thing before, exactly 5 times |
| `[abc]` | one of `a`, `b`, `c` |
| `( )` | a group you want to get out |

Write patterns as raw strings, `r"..."`, so the backslashes stay as they are. `re.findall` gives every match as a list:

In [ ]:
import re

print(re.findall(r"\d+", "move 13 from 2 to 7"))

**Predict:**

In [ ]:
print([int(x) for x in re.findall(r"\d+", "x=-3, y=12")])

In [ ]:
print([int(x) for x in re.findall(r"-?\d+", "x=-3, y=12")])

`search` finds the first match; groups take parts out of it:

In [ ]:
match = re.search(r"(\d\d):(\d\d)", "The train leaves at 07:42 from platform 3")
print(match.group(0))
print(match.groups())

Without a match, `search` gives `None`, not an error, so check before calling `.group()`:

In [ ]:
print(re.search(r"(\d\d):(\d\d)", "no time here"))

`fullmatch` checks the *whole* string, e.g. to validate input in your web app. **Predict:**

In [ ]:
def is_station_id(text):
    return re.fullmatch(r"\d{5}", text) is not None


print(is_station_id("05906"), is_station_id("5906"), is_station_id("059061"))

In [ ]:
print(re.search(r"\d{5}", "059061"))

`re.sub` replaces every match:

In [ ]:
print(re.sub(r"\s+", " ", "too    many  \t spaces"))

Regular expressions get unreadable quickly. Keep them short, and test them like any other function.
https://regex101.com explains a pattern piece by piece.

## `datetime` and `zoneinfo`

In Ex 7, pandas turned `20251124` into a date for you. In plain Python:

In [ ]:
from datetime import UTC, date, datetime, timedelta

exam = date(2026, 12, 18)
today = date(2026, 11, 25)
print(exam - today)

In [ ]:
print((exam - today).days)
print(today + timedelta(weeks=3))
print(exam.weekday())  # Monday is 0
print(exam.strftime("%A, %d %B %Y"))

Reading dates from text: `strptime` ("string parse time") with a format, or ISO format directly.

In [ ]:
print(datetime.strptime("20251124", "%Y%m%d"))
print(date.fromisoformat("2026-11-25"))
print(date(2026, 11, 25).isoformat())

### Time zones

A `datetime` without a time zone is *naive*: 14:00, but where? `zoneinfo` knows the world's time
zones, including when they switch to and from summer time.

In [ ]:
from zoneinfo import ZoneInfo

berlin = ZoneInfo("Europe/Berlin")
meeting = datetime(2026, 11, 25, 14, 0, tzinfo=berlin)
print(meeting)
print(meeting.astimezone(ZoneInfo("Asia/Tokyo")))
print(meeting.astimezone(UTC))

In [ ]:
naive = datetime(2026, 11, 25, 14, 0)
print(naive.tzinfo)
print(meeting - naive)

In Germany, the clocks went back an hour on Sunday, Oct 25, 2026 (3:00 became 2:00).
How long was it from Saturday noon to Sunday noon?

In [ ]:
saturday = datetime(2026, 10, 24, 12, 0, tzinfo=berlin)
sunday = datetime(2026, 10, 25, 12, 0, tzinfo=berlin)
print(sunday - saturday)

In [ ]:
print(sunday.astimezone(UTC) - saturday.astimezone(UTC))

With both times in the same zone, Python subtracts the clock times: one day. The time that
actually passed was 25 hours. Hourly data (electricity, air quality) has 25-hour and 23-hour days.
Safest: compute in UTC, and convert to local time only to show it. The current time in UTC is
`datetime.now(UTC)`.

## `pickle`: saving Python objects

`json` (week 4) saves dicts, lists, strings and numbers. For the project you may want to train a
model once and load it in your web app. `pickle` saves almost any Python object as bytes.

In [ ]:
import json
import pickle
from dataclasses import dataclass


@dataclass
class Station:
    name: str
    height: int


data = {"stations": [Station("Mannheim", 96)], "created": date(2026, 11, 25)}
blob = pickle.dumps(data)
print(blob[:40])
print(pickle.loads(blob))

In [ ]:
print(json.dumps(data))

In [ ]:
from pathlib import Path

path = Path("stations.pickle")
path.write_bytes(pickle.dumps(data))
print(pickle.loads(path.read_bytes()))
path.unlink()

### Loading a pickle runs code

A pickle stores instructions for rebuilding the object. Anyone can write such instructions:

In [ ]:
class NotAStation:
    def __reduce__(self):
        return (print, ("This ran while loading!",))


evil = pickle.dumps(NotAStation())
del NotAStation
pickle.loads(evil)

`__reduce__` tells `pickle` how to rebuild the object: "call `print` with these arguments". It
could be any function, e.g. one that deletes files, and the class doesn't even need to exist when
the pickle is loaded. **Only load pickles you made yourself.** That includes models downloaded from
the internet. Pickles also break when you change the class they contain.

## `logging`: `print` for programs that keep running

A web app runs for hours; you want to know what happened without deleting `print`s afterwards.
Every message has a level: `DEBUG` < `INFO` < `WARNING` < `ERROR` < `CRITICAL`, and you choose
from which level on messages are shown. FastAPI's server logs this way too.

**Predict:** which lines appear?

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True
)
log = logging.getLogger("weather")

log.debug("parsed line 17")
log.info("loaded 89474 rows")
log.warning("station 5906: 312 days missing")

In [ ]:
log.setLevel(logging.DEBUG)
log.debug("parsed line 18")

`log.exception` inside `except` adds the traceback:

In [ ]:
try:
    rain_per_day = 120 / 0
except ZeroDivisionError:
    log.exception("could not compute the average")
print("the program goes on")

## `random` and `secrets`

On Nov 18, `random.Random(1)` shuffled the Titanic passengers the same way every time.
**Predict:** what's the difference between the two lines?

In [ ]:
import random

rng = random.Random(42)
print(rng.randint(1, 6), rng.choice(["rock", "paper", "scissors"]), rng.sample(range(100), 3))
rng = random.Random(42)
print(rng.randint(1, 6), rng.choice(["rock", "paper", "scissors"]), rng.sample(range(100), 3))

Predictable is what you want for experiments and tests. For passwords, login tokens or API keys it's
exactly wrong: use `secrets`.

In [ ]:
import secrets

print(secrets.token_urlsafe(16))

## What else exists

- **Numbers:** `math` (`isqrt`, `inf`, `isclose`), `statistics` (mean, median for small data),
  `fractions`, `decimal` (exact decimal numbers, e.g. for money; remember `0.1 + 0.2` from week 1)
- **Files:** `shutil` (copy, move, delete folders), `zipfile`, `gzip`, `tempfile`
- **Text:** `textwrap`, `string`, `difflib`
- **Data:** `sqlite3` (a database in a single file), `bisect` (search in a sorted list), `enum`
- **Running programs:** `argparse` (command line options), `subprocess` (start other programs),
  `os`, `sys`
- **Internet:** `urllib.request`, `http.server`, `email`, `hashlib`, `uuid`. (FastAPI on Friday is
  *not* part of the standard library; it's installed with `uv add`.)
- **Development:** `timeit`, `pdb` (the debugger), `typing`, `unittest` (we use `pytest`)

Three small examples:

In [ ]:
import difflib
import statistics

print(0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3))
print(statistics.median([4.1, 5.3, 2.0, 2.0, 6.5]))
print(difflib.get_close_matches("Manheim", ["Mannheim", "Mainz", "Heidelberg", "Mosbach"]))

**Where to look:** https://docs.python.org/3/library/, `help(module)`, and https://pymotw.com/3/
(an example for every module). Asking an AI "is there something in the standard library for this?"
is an explanation question, so it's fine at level 1.